# Reporting Views — Hospitality Analytics
Creates SQL views over Gold tables for Power BI Direct Lake connectivity: executive summary, revenue heat-map, loyalty analysis, and property performance.

In [ ]:
# vw_executive_summary — top-line KPIs per property
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    s.property_id,
    s.property_name,
    s.city,
    s.country,
    s.star_rating,
    s.property_type,
    s.total_bookings,
    s.total_revenue,
    s.adr,
    s.avg_los,
    s.revpar,
    s.repeat_guest_pct,
    s.review_count,
    s.avg_overall_score,
    s.performance_band,
    RANK() OVER (ORDER BY s.revpar DESC NULLS LAST)    AS revpar_rank,
    RANK() OVER (ORDER BY s.avg_overall_score DESC NULLS LAST) AS satisfaction_rank
FROM gold_property_scorecards s
""")
print('Created: vw_executive_summary')

In [ ]:
# vw_revenue_heatmap — daily RevPAR by city for trend/calendar views
spark.sql("""
CREATE OR REPLACE VIEW vw_revenue_heatmap AS
SELECT
    check_in_date       AS stay_date,
    city,
    country,
    star_rating,
    SUM(bookings)       AS total_bookings,
    SUM(total_revenue)  AS total_revenue,
    ROUND(AVG(adr), 2)  AS avg_adr,
    ROUND(AVG(revpar), 2) AS avg_revpar,
    ROUND(AVG(occupancy_rate), 2) AS avg_occupancy_rate
FROM gold_daily_revenue
GROUP BY check_in_date, city, country, star_rating
""")
print('Created: vw_revenue_heatmap')

In [ ]:
# vw_loyalty_analysis — loyalty tier KPIs for guest lifecycle reporting
spark.sql("""
CREATE OR REPLACE VIEW vw_loyalty_analysis AS
SELECT
    loyalty_tier,
    region,
    SUM(total_bookings)         AS total_bookings,
    SUM(unique_guests)          AS unique_guests,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(AVG(avg_rate), 2)     AS avg_rate,
    ROUND(AVG(avg_los), 2)      AS avg_los,
    ROUND(AVG(repeat_guest_pct), 1) AS repeat_guest_pct,
    ROUND(SUM(total_revenue) / NULLIF(SUM(unique_guests), 0), 2) AS revenue_per_guest
FROM gold_loyalty_segments
GROUP BY loyalty_tier, region
""")
print('Created: vw_loyalty_analysis')

In [ ]:
# vw_property_performance — full scorecard view for property comparison matrix
spark.sql("""
CREATE OR REPLACE VIEW vw_property_performance AS
SELECT
    property_id,
    property_name,
    city,
    country,
    star_rating,
    property_type,
    room_count,
    total_bookings,
    ROUND(total_revenue, 2)   AS total_revenue,
    adr,
    revpar,
    avg_los,
    repeat_guest_pct,
    review_count,
    avg_overall_score,
    avg_cleanliness,
    avg_service,
    avg_value,
    avg_food,
    performance_band
FROM gold_property_scorecards
""")
print('Created: vw_property_performance')

In [ ]:
# Validate
views = ['vw_executive_summary', 'vw_revenue_heatmap', 'vw_loyalty_analysis', 'vw_property_performance']
for v in views:
    n = spark.sql(f'SELECT COUNT(*) AS n FROM {v}').collect()[0]['n']
    print(f'  {v:<35} {n:>6,} rows')